In [ ]:
# Setup and data download from WRDS

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import wrds
from statsmodels.tsa.api import VAR
from tqdm import tqdm

os.environ['SQLALCHEMY_POOL_TIMEOUT'] = '60'

# Find the Thesis project root so the notebook works from either Thesis/ or Thesis/Scripts/.
candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_DIR = next(
    (root for root in candidate_roots if (root / "Data").exists() and (root / "Scripts").exists()),
    Path.cwd().resolve(),
)
DATA_DIR = PROJECT_DIR / "Data"
OUTPUTS_DIR = PROJECT_DIR / "Outputs"

# Make sure the expected folders exist before reading or writing files.
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------
# Parameters
# ------------------------
START_YEAR = 1997
END_YEAR = 2000
MIN_OBS = 50      # minimum daily obs per (permno, year) for VAR
LAGS = 5          # VAR lag length, as in Brogaard
HORIZON = 15      # IRF horizon

# Store the cached CRSP file inside Thesis/Data.
CRSP_CACHE_FILE = DATA_DIR / "CRSP.csv"

# Check if data already exists locally.
if CRSP_CACHE_FILE.exists():
    print(f"Loading cached CRSP data from {CRSP_CACHE_FILE} ...")
    crsp = pd.read_csv(CRSP_CACHE_FILE, parse_dates=['date'])
    print(f"Loaded CRSP shape: {crsp.shape}")
else:
    print("Cached data not found. Downloading CRSP daily data from WRDS ...")

    # Retry the WRDS connection a few times because remote logins can fail intermittently.
    max_retries = 3
    db = None
    for attempt in range(max_retries):
        try:
            db = wrds.Connection()
            print("Connected to WRDS successfully")
            break
        except Exception as error:
            print(f"Connection attempt {attempt + 1} failed: {error}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print("Max retries reached. Check your connection and credentials.")
                raise

    # Download CRSP daily stock and market data for the requested years.
    query = f"""
        select
            a.permno,
            a.date,
            a.ret,
            a.prc,
            a.vol,
            c.sharetype,
            c.securitytype,
            c.securitysubtype,
            c.primaryexch,
            c.conditionaltype,
            b.ewretd
        from crsp.dsf as a
        left join crsp.dsi as b
            on a.date = b.date
        left join crsp.stksecurityinfohist as c
            on a.permno = c.permno
            and a.date >= c.begdt
            and a.date <= c.enddt
        where c.sharetype = 'NS'
            and c.securitytype = 'EQTY'
            and c.securitysubtype = 'COM'
            and c.primaryexch in ('N', 'A', 'Q')
            and c.conditionaltype = 'RW'
            and a.date between '{START_YEAR}-01-01' and '{END_YEAR}-12-31'
    """

    crsp = db.raw_sql(query, date_cols=['date'])

    print("Raw CRSP shape:", crsp.shape)

    # Remove rows that cannot be used in the decomposition.
    crsp = crsp.dropna(subset=['ret', 'prc', 'vol', 'ewretd'])

    # Save the raw file inside Thesis/Data for reuse.
    crsp.to_csv(CRSP_CACHE_FILE, index=False)
    print(f"CRSP data saved to {CRSP_CACHE_FILE}")

# Add the year because the next cells decompose variance one stock-year at a time.
crsp['year'] = crsp['date'].dt.year

Cached data not found. Downloading CRSP daily data from WRDS ...
Connection attempt 1 failed: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: SSL connection has been closed unexpectedly

(Background on this error at: https://sqlalche.me/e/20/e3q8)
Retrying in 1 seconds...
Connection attempt 2 failed: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "jauder"

(Background on this error at: https://sqlalche.me/e/20/e3q8)
Retrying in 2 seconds...
Connection attempt 3 failed: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "jauder"

(Background on this error at: https://sqlalche.me/e/20/e3q8)
Max retries reached. Check your connection and credentials.


Cached data not found. Downloading CRSP daily data from WRDS ...
Connection attempt 1 failed: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: SSL connection has been closed unexpectedly

(Background on this error at: https://sqlalche.me/e/20/e3q8)
Retrying in 1 seconds...
Connection attempt 2 failed: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "jauder"

(Background on this error at: https://sqlalche.me/e/20/e3q8)
Retrying in 2 seconds...
Connection attempt 3 failed: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "jauder"

(Background on this error at: https://sqlalche.me/e/20/e3q8)
Max retries reached. Check your connection and credentials.


OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "jauder"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# Construct VAR variables and winsorize

# Market return (equal-weighted) in basis points
crsp['rm'] = crsp['ewretd'].astype(float) * 10000.0

# Stock return in basis points
crsp['r'] = crsp['ret'].astype(float) * 10000.0

# Signed dollar volume in $ thousands
# CRSP prc can be signed; use abs(prc) for dollar volume
crsp['dollar_vol'] = np.abs(crsp['prc'].astype(float)) * crsp['vol'].astype(float)
crsp['sign_ret']   = np.sign(crsp['ret'].astype(float).fillna(0.0))
crsp['x']          = crsp['dollar_vol'] * crsp['sign_ret'] / 1000.0

# Keep necessary columns
crsp = crsp[['permno', 'date', 'year', 'rm', 'x', 'r']].dropna()

# ------------------------
# Winsorize rm, x, r at 5% and 95% globally
# ------------------------
def winsorize_series(s, lower=0.05, upper=0.95):
    lo, hi = s.quantile([lower, upper])
    return s.clip(lower=lo, upper=hi)

for col in ['rm', 'x', 'r']:
    crsp[col] = winsorize_series(crsp[col])

In [ ]:
# Core Brogaard decomposition for a single stock–year
def brogaard_decomposition_one_block(df_block,
                                     lags=LAGS,
                                     horizon=HORIZON,
                                     min_obs=MIN_OBS):
    """
    Perform Brogaard et al. variance decomposition on a single stock-year block.

    Parameters
    ----------
    df_block : DataFrame with columns ['date','rm','x','r'] for one (permno, year)
    Returns
    -------
    dict with variance components and shares, or None if VAR fails / too few obs.
    """
    df_block = df_block.sort_values('date').copy()
    n = df_block.shape[0]
    if n < min_obs:
        return None

    # Endogenous variables in order [rm, x, r]
    Y = df_block[['rm', 'x', 'r']].astype(float)

    # ------------------------
    # Step 1: Estimate reduced-form VAR(p)
    # ------------------------
    try:
        model = VAR(Y)
        res   = model.fit(maxlags=lags, ic=None, trend='c')
    except Exception:
        return None

    # Enforce the exact lag length
    if res.k_ar != lags:
        # If needed, refit with fixed lags
        try:
            res = model.fit(lags, trend='c')
        except Exception:
            return None

    # Effective number of observations used in VAR
    n_eff = res.nobs
    if n_eff < min_obs:
        return None

    # Reduced-form residuals e_t: shape (n_eff, k)
    resid = res.resid
    k = resid.shape[1]

    # Degrees-of-freedom‑adjusted covariance of residuals Σ_e
    # (unbiased sample covariance)
    sigma_e = np.cov(resid.T, ddof=1)

    # ------------------------
    # Structural identification: epsilon_t = B * e_t
    # Use Cholesky-style identification as in SVAR with lower-triangular A/B
    # Here choose B = (chol(Σ_e))^{-1}, so Σ_ε = I (uncorrelated shocks).
    # ------------------------
    try:
        P = np.linalg.cholesky(sigma_e)   # Σ_e = P P'
    except np.linalg.LinAlgError:
        # Non positive-definite covariance; skip
        return None

    B = np.linalg.inv(P)
    sigma_epsilon = B @ sigma_e @ B.T   # should be close to identity

    # Structural shocks epsilon_t: (n_eff, k)
    epsilons = (B @ resid.T).T

    # ------------------------
    # Step 2: 15-step cumulative structural IRFs (un-orthogonalized)
    # ------------------------
    # res.coefs has shape (lags, k, k): A_j matrices
    # A_j is effect of y_{t-j} on y_t.
    coefs = res.coefs  # array of shape (lags, k, k)

    # Build A_1,...,A_p matrices as list
    A_list = [coefs[j] for j in range(lags)]  # each is (k, k)

    # IRFs Φ_i, i = 0,...,HORIZON
    Phi = []
    Phi_0 = np.eye(k)
    Phi.append(Phi_0)

    # Compute Φ_i by recursion:
    # Φ_i = sum_{j=1}^{min(i,p)} Φ_{i-j} A_j
    for i in range(1, horizon + 1):
        Phi_i = np.zeros((k, k))
        max_j = min(i, lags)
        for j in range(1, max_j + 1):
            Phi_i += Phi[i - j] @ A_list[j - 1]
        Phi.append(Phi_i)

    # Structural IRFs Λ_i = Φ_i B^{-1}
    B_inv = np.linalg.inv(B)
    Lambda = [Phi_i @ B_inv for Phi_i in Phi[1:]]  # 1..HORIZON

    # Cumulative un-orthogonalized structural IRFs:
    csirf = sum(Lambda)   # (k, k)

    # θ for stock returns = 3rd row (index 2) of csirf
    theta = csirf[2, :]   # [θ_rm, θ_x, θ_r]

    # ------------------------
    # Step 3: Noise term
    # Δs* = r_t - θ_rm * ε_rm,t - θ_x * ε_x,t - θ_r * ε_r,t
    # ------------------------
    # Align epsilons with the VAR sample: res.yendog / res.resid index
    r_var_sample = res.endog[:, 2]  # stock return r in VAR sample order

    delta_s = r_var_sample - (
        theta[0] * epsilons[:, 0] +
        theta[1] * epsilons[:, 1] +
        theta[2] * epsilons[:, 2]
    )

    # Variance of noise (unbiased)
    sigma_s2 = float(np.var(delta_s, ddof=1))

    # ------------------------
    # Step 4: Information-based variance components
    # MktInfo = θ_rm^2 * var(ε_rm)
    # PrivateInfo = θ_x^2 * var(ε_x)
    # PublicInfo = θ_r^2 * var(ε_r)
    # ------------------------
    var_eps = np.diag(sigma_epsilon)  # variances of structural shocks

    mkt_info_var     = float(theta[0] ** 2 * var_eps[0])
    private_info_var = float(theta[1] ** 2 * var_eps[1])
    public_info_var  = float(theta[2] ** 2 * var_eps[2])

    sigma_w2 = mkt_info_var + private_info_var + public_info_var
    total_var = sigma_w2 + sigma_s2

    if total_var <= 0 or not np.isfinite(total_var):
        return None

    # ------------------------
    # Step 5: Variance shares
    # ------------------------
    mkt_share     = mkt_info_var     / total_var
    private_share = private_info_var / total_var
    public_share  = public_info_var  / total_var
    noise_share   = sigma_s2         / total_var

    result = {
        'n_obs_var'          : int(n_eff),
        'theta_rm'           : float(theta[0]),
        'theta_x'            : float(theta[1]),
        'theta_r'            : float(theta[2]),
        'var_eps_rm'         : float(var_eps[0]),
        'var_eps_x'          : float(var_eps[1]),
        'var_eps_r'          : float(var_eps[2]),
        'mkt_info_var'       : mkt_info_var,
        'private_info_var'   : private_info_var,
        'public_info_var'    : public_info_var,
        'noise_var'          : sigma_s2,
        'sigma_w2'           : sigma_w2,
        'total_var'          : total_var,
        'mkt_info_share'     : mkt_share,
        'private_info_share' : private_share,
        'public_info_share'  : public_share,
        'noise_share'        : noise_share,
    }

    return result

In [ ]:
# Loop over (permno, year) and build the panel of decompositions.

results = []
grouped = crsp.groupby(['permno', 'year'])

print("Running Brogaard decomposition by (permno, year) ...")
for (permno, year), df_block in tqdm(grouped, total=len(grouped)):
    out = brogaard_decomposition_one_block(df_block)
    if out is not None:
        out['permno'] = permno
        out['year'] = year
        results.append(out)

brogaard_panel = pd.DataFrame(results)

print("Decomposition results shape:", brogaard_panel.shape)
print(brogaard_panel.head())

# Save the panel inside Thesis/Outputs to keep generated files out of the project root.
output_path = OUTPUTS_DIR / "brogaard_decomposition_panel.csv"
brogaard_panel.to_csv(output_path, index=False)
print(f"Saved decomposition panel to {output_path}")